In [ ]:

import pandas as pd
import yfinance as yf
from edgar import *
from IPython.display import display
import os
from dotenv import load_dotenv
load_dotenv()
set_identity(os.getenv("EMAIL"))
print("All imported")



In [4]:
def get_companys_datas(acq, tgt, verbose= False):

    acq_dict = yf.Ticker(acq).info
    tgt_dict = yf.Ticker(tgt).info
    
    if verbose == False: 
        pass
    else:
        print(f"{"values":<15} | {acq_dict["symbol"]:<25} | {tgt_dict["symbol"]:<25}")
        print("-"*60)
        print(f"{"Market cap":<15} | {round(acq_dict["marketCap"]):<25,} | {round(tgt_dict["marketCap"]):<25,}")
        print(f"{"Price":<15} | {acq_dict["previousClose"]:<25} | {tgt_dict["previousClose"]:<25}")
        print(f"{"Shares":<15} | {acq_dict["sharesOutstanding"]:<25} | {tgt_dict["sharesOutstanding"]:<25}\n\n")

    return acq_dict, tgt_dict

In [5]:
def contract_offer(acq_dict, tgt_dict, offer_premium=.60, stock_pct=0.50, tax_rate=0.40,
                   years= 5, interest_rate= 0.05, financing_fees_pct=.035,transaccion_fees_pct= .02, 
                   synergies_pct=0, amortization_years= 10, verbose=False ):
    
    # % of deal pay by cahs and with stocks
    cash_pct= 1-stock_pct
    
    # Amount of money the acusition will cost
    share_price= tgt_dict["previousClose"]* (1+offer_premium)
    offer_value= tgt_dict["sharesOutstanding"]* share_price
    acq_issued_shares= offer_value/acq_dict["previousClose"] * stock_pct
    
    # All expenses and writre offs 
    #Transaccion fees:
    transaccion_fees= offer_value * transaccion_fees_pct 
    #Financing fees:
    acq_borrowing= offer_value * cash_pct
    financing_fees= acq_borrowing * financing_fees_pct 
    financing_fees_amort= financing_fees/years
    #Posible write of 
    synergies= tgt_dict["totalRevenue"]* synergies_pct
    asset_write_off= (offer_value - tgt_dict["bookValue"]) * 0.15 
    incremental_DA_expense = asset_write_off/amortization_years 
    
    # Expected earnings
    acq_implide_net_inc= acq_dict["sharesOutstanding"]* acq_dict["epsCurrentYear"]
    tgt_implide_net_inc= tgt_dict["sharesOutstanding"]* tgt_dict["epsCurrentYear"]
    acq_implide_pretax_inc= acq_implide_net_inc /(1-tax_rate)
    tgt_implide_pretax_inc= tgt_implide_net_inc /(1-tax_rate)
    
    #Getting all together and in negatice (accounting reasons)
    profroma_pretax_unadj= acq_implide_pretax_inc + tgt_implide_pretax_inc
    interest_expense_deal= acq_borrowing * interest_rate * -1
    incremental_DA_expense *= -1
    transaccion_fees *= -1
    financing_fees_amort *=-1
    synergies *=1 
    #after merge
    profroma_pretax_adj= profroma_pretax_unadj + interest_expense_deal + incremental_DA_expense + transaccion_fees + financing_fees_amort + synergies
    proforma_net_income= profroma_pretax_adj * (1-tax_rate)
    proforma_shares_outstanding= acq_dict["sharesOutstanding"]+ acq_issued_shares
    proforma_eps= proforma_net_income/proforma_shares_outstanding
    accretion_dilution_per_share= proforma_eps - acq_dict["epsCurrentYear"]
    accretion_dilution_pct= (proforma_eps / acq_dict["epsCurrentYear"])-1
    
    if verbose == False:
        pass
    else: 
        print("Parameters:")
        print(f"Offer premium:            | {offer_premium*100}%")
        print(f"% of cash:                | {cash_pct*100}%")
        print(f"% of stock:               | {stock_pct*100}%")
        print(f"tax rate:                 | {tax_rate*100}%")
        print(f"interest rate:            | {interest_rate*100}%")
        print(f"% financing fees:         | {transaccion_fees_pct*100}%")
        print(f"% transaccion fees:       | {transaccion_fees_pct*100}%")
        print(f"% synergies:              | {synergies_pct*100}%")
        print(f"{"-"*60}\n")
        
        print("Deal:")
        print(f"Share price:              | {round(share_price):,}")
        print(f"Offer Value:              | {round(offer_value):,}")
        print(f"Money borrowed:           | {round(acq_borrowing):,}")
        print(f"Financing fees:           | {round(financing_fees):,}")
        print(f"Shares issued:            | {round(acq_issued_shares):,}\n")
        print(f"{"-"*60}\n")
        
        print("Income:")
        print(f"Accuary net income:       | {round(acq_implide_net_inc):,}")
        print(f"Target net income:        | {round(tgt_implide_net_inc):,}")
        print(f"Accuary pre tax income:   | {round(acq_implide_pretax_inc):,}")
        print(f"Target pre tax income:    | {round(tgt_implide_pretax_inc):,}")
        print(f"{"-"*60}\n")
        
        print("Totals")
        print(f"Proforma pretax unadj:    | {round(profroma_pretax_unadj):,}")
        print(f"Interest expenses:        | ({round(interest_expense_deal*-1):,})")
        print(f"Amort of finance fees:    | ({round(financing_fees_amort*-1):,})")
        print(f"Transaccion fees:         | ({round(transaccion_fees*-1):,})")
        print(f"D/A write off:            | ({round(incremental_DA_expense*-1):,})")
        print(f"% synergies:              | {round(synergies):,}")
        print(f"{"-"*60}\n")
        
        print("After merge:")
        print(f"Proforma pretax adj:      | {round(profroma_pretax_adj):,}")
        print(f"Proforma Net:             | {round(proforma_net_income):,}")
        print(f"Proforma shares:          | {round(proforma_shares_outstanding):,}")
        print(f"Proforma eps:             | {round(proforma_eps,ndigits=2)}")
        print(f"{"-"*60}\n")
        
        print("Results")
        print(f"Accretion/Dilution:       | $ {accretion_dilution_per_share:.2f}")
        print(f"%Accretion/Dilution:      | % {accretion_dilution_pct:.2f}")
    
    return accretion_dilution_pct

In [6]:
def highlight_irr_accdil(val):
    if val >= 0:
        return "background-color: #d4edda !important; color: black !important"
    elif val >= -.05:
        return "background-color: #fff3cd !important; color: black !important"
    else:
        return "background-color: #f8d7da !important; color: black !important"

In [13]:
def sensitivity_accretion_dilution(acq, tgt,verbose= False, steps= 1):
    
    rows= []
    for i in range (0, 11, steps):
        row= []
        for j in range(0, 11, steps):

            accretion_dilution_pct= round(contract_offer(acq, tgt, offer_premium=i/10, stock_pct= j/10, verbose=False), ndigits= 2)
            
            row.append(accretion_dilution_pct)
            
        rows.append(row)
    
    df= df = pd.DataFrame(rows, columns = [f"Stock {x*10}%" for x in range(0, 11, steps)], index   = [f"Offer premium {x*10}%"  for x in range(0, 11, steps)])
    styled= df.style.format("{:.2f}%").map(highlight_irr_accdil)
    display(styled)
    
    return df

In [14]:
def accretion_dilution_model(acq,tgt):
    acq_dict, tgt_dict = get_companys_datas(acq,tgt, verbose= True)
    contract_offer(acq_dict, tgt_dict, verbose=True )
    sensitivity_accretion_dilution(acq_dict, tgt_dict)

In [15]:
accretion_dilution_model("NCLH","SONO")

values          | NCLH                      | SONO                     
------------------------------------------------------------
Market cap      | 8,956,026,880             | 1,629,363,328            
Price           | 20.12                     | 13.59                    
Shares          | 455545641                 | 120872657                


Parameters:
Offer premium:            | 60.0%
% of cash:                | 50.0%
% of stock:               | 50.0%
tax rate:                 | 40.0%
interest rate:            | 5.0%
% financing fees:         | 2.0%
% transaccion fees:       | 2.0%
% synergies:              | 0%
------------------------------------------------------------

Deal:
Share price:              | 22
Offer Value:              | 2,628,255,054
Money borrowed:           | 1,314,127,527
Financing fees:           | 45,994,463
Shares issued:            | 65,314,489

------------------------------------------------------------

Income:
Accuary net income:       | 1,063,935,9

,Stock 0%,Stock 10%,Stock 20%,Stock 30%,Stock 40%,Stock 50%,Stock 60%,Stock 70%,Stock 80%,Stock 90%,Stock 100%
Offer premium 0%,0.04%,0.03%,0.02%,0.00%,-0.01%,-0.02%,-0.03%,-0.04%,-0.05%,-0.06%,-0.07%
Offer premium 10%,0.03%,0.02%,0.00%,-0.01%,-0.02%,-0.03%,-0.05%,-0.06%,-0.07%,-0.08%,-0.09%
Offer premium 20%,0.02%,0.01%,-0.01%,-0.02%,-0.03%,-0.05%,-0.06%,-0.07%,-0.08%,-0.09%,-0.11%
Offer premium 30%,0.02%,-0.00%,-0.02%,-0.03%,-0.05%,-0.06%,-0.07%,-0.09%,-0.10%,-0.11%,-0.12%
Offer premium 40%,0.01%,-0.01%,-0.03%,-0.04%,-0.06%,-0.07%,-0.09%,-0.10%,-0.11%,-0.12%,-0.14%
Offer premium 50%,-0.00%,-0.02%,-0.04%,-0.05%,-0.07%,-0.09%,-0.10%,-0.11%,-0.13%,-0.14%,-0.15%
Offer premium 60%,-0.01%,-0.03%,-0.05%,-0.07%,-0.08%,-0.10%,-0.11%,-0.13%,-0.14%,-0.15%,-0.17%
Offer premium 70%,-0.02%,-0.04%,-0.06%,-0.08%,-0.09%,-0.11%,-0.12%,-0.14%,-0.15%,-0.17%,-0.18%
Offer premium 80%,-0.03%,-0.05%,-0.07%,-0.09%,-0.10%,-0.12%,-0.14%,-0.15%,-0.17%,-0.18%,-0.19%
Offer premium 90%,-0.04%,-0.06%,-0.08%,-0.10%,-0.12%,-0.13%,-0.15%,-0.16%,-0.18%,-0.19%,-0.21%


In [10]:
accretion_dilution_model("TSLA","NVDA")

values          | TSLA                      | NVDA                     
------------------------------------------------------------
Market cap      | 1,437,294,002,176         | 4,258,235,940,864        
Price           | 380.85                    | 175.68                   
Shares          | 3752431984                | 24300000000              


Parameters:
Offer premium:            | 60.0%
% of cash:                | 50.0%
% of stock:               | 50.0%
tax rate:                 | 40.0%
interest rate:            | 5.0%
% financing fees:         | 2.0%
% transaccion fees:       | 2.0%
% synergies:              | 0%
------------------------------------------------------------

Deal:
Share price:              | 281
Offer Value:              | 6,830,438,400,000
Money borrowed:           | 3,415,219,200,000
Financing fees:           | 119,532,672,000
Shares issued:            | 8,967,360,378

------------------------------------------------------------

Income:
Accuary net income:   

,Stock 0%,Stock 10%,Stock 20%,Stock 30%,Stock 40%,Stock 50%,Stock 60%,Stock 70%,Stock 80%,Stock 90%,Stock 100%
Offer premium 0%,-4.37%,-2.15%,-0.77%,0.18%,0.87%,1.40%,1.81%,2.15%,2.42%,2.65%,2.85%
Offer premium 10%,-7.39%,-4.26%,-2.37%,-1.11%,-0.20%,0.48%,1.00%,1.43%,1.78%,2.07%,2.31%
Offer premium 20%,-10.40%,-6.27%,-3.86%,-2.29%,-1.18%,-0.35%,0.29%,0.80%,1.21%,1.55%,1.84%
Offer premium 30%,-13.42%,-8.20%,-5.26%,-3.37%,-2.06%,-1.09%,-0.35%,0.24%,0.71%,1.10%,1.43%
Offer premium 40%,-16.44%,-10.04%,-6.56%,-4.37%,-2.86%,-1.76%,-0.92%,-0.26%,0.27%,0.71%,1.07%
Offer premium 50%,-19.46%,-11.81%,-7.78%,-5.28%,-3.59%,-2.37%,-1.44%,-0.72%,-0.13%,0.35%,0.75%
Offer premium 60%,-22.48%,-13.51%,-8.92%,-6.14%,-4.27%,-2.92%,-1.91%,-1.12%,-0.49%,0.03%,0.46%
Offer premium 70%,-25.50%,-15.14%,-10.00%,-6.93%,-4.89%,-3.43%,-2.34%,-1.49%,-0.81%,-0.26%,0.20%
Offer premium 80%,-28.52%,-16.71%,-11.01%,-7.67%,-5.46%,-3.90%,-2.73%,-1.83%,-1.11%,-0.52%,-0.04%
Offer premium 90%,-31.54%,-18.21%,-11.97%,-8.35%,-5.99%,-4.33%,-3.09%,-2.14%,-1.38%,-0.76%,-0.25%
